# End-to-End Flow of Qdrant

This chapter explains how a production RAG application stores, searches, filters, and secures document embeddings using Qdrant.

## Topic 1: Why Do We Need a Vector Database?

### Why Do We Need It?

When building a RAG application, documents must be searched based on **meaning**, not exact keywords.

Traditional databases like SQL perform keyword search but cannot understand semantic similarity.

To perform semantic search, every document is converted into an embedding, and those embeddings need a specialized database for efficient similarity search.

---

### What Is It?

A Vector Database is a database optimized for storing vector embeddings and performing **Nearest Neighbor Search (Similarity Search).**

Unlike SQL databases, it retrieves documents based on semantic similarity rather than exact text matching.

---

### Example

Knowledge Base

- FD Interest Rate
- FD Premature Closure
- Home Loan EMI
- Credit Card Rewards

Customer asks:

> What is the penalty for closing my Fixed Deposit early?

Although none of the documents contain the exact sentence, the Vector Database retrieves **FD Premature Closure** because both have similar meanings.

---

### How Does It Work?

Step 1

An embedding model converts every document into a vector.

Step 2

The vectors are stored inside the Vector Database.

Step 3

The user's query is converted into another vector.

Step 4

The Vector Database compares the query vector with stored vectors.

Step 5

The nearest vectors are returned.

Step 6

The application retrieves the original documents.

---

### Real-World Issues, Edge Cases & Debugging

- Poor embedding models produce poor search quality.
- Large documents should be chunked before embedding.
- Duplicate chunks reduce retrieval quality.
- Different embedding models generate incompatible vectors.

---

### Common Mistakes

- Expecting SQL to perform semantic search.
- Storing raw documents instead of embeddings.
- Using different embedding models for indexing and querying.

---

## Topic 2: Why Are There Different Vector Databases?

### Why Do We Need It?

Once we know we need a Vector Database, the next question is:

**Which Vector Database should we use?**

Different databases provide different capabilities.

---

### What Is It?

Common Vector Databases include:

- FAISS
- Chroma
- Qdrant
- Pinecone
- Weaviate

Some are simple search libraries, while others are complete production databases.

---

### Example

FAISS

Stores:

- Vector IDs
- Embeddings

Application stores:

- Original documents
- Metadata

Qdrant

Stores:

- Vector IDs
- Embeddings
- Metadata (Payload)
- Collections
- Persistent storage

---

### How Does It Work?

Documents are converted into embeddings.

The Vector Database stores those embeddings.

During search, the database returns the IDs of the nearest vectors.

The application uses those IDs to retrieve the original documents.

---

### Real-World Issues, Edge Cases & Debugging

- FAISS cannot perform metadata filtering.
- Cloud databases require internet connectivity.
- Local databases simplify development.
- Some databases support persistence while others require additional implementation.

---

### Common Mistakes

- Assuming every Vector Database provides the same features.
- Expecting FAISS to store documents.
- Choosing a production database for a learning project.

---

## Topic 3: Why Did We Choose Qdrant?

### Why Do We Need It?

After comparing Vector Databases, we must choose one that satisfies our project requirements.

---

### What Is It?

Qdrant is an open-source, production-ready Vector Database designed for semantic search.

---

### Example

Our project required:

- Easy to learn
- Local execution
- Metadata filtering
- Persistent storage
- Production deployment
- Open source

Qdrant satisfies all these requirements.

---

### How Does It Work?

Development uses:

```python
QdrantClient(":memory:")
```

Production uses:

```python
QdrantClient(url="http://localhost:6333")
```

The application logic remains unchanged.

Only the connection changes.

---

### Real-World Issues, Edge Cases & Debugging

- Using `:memory:` in production.
- Forgetting Docker volumes.
- Recreating collections after embedding dimension changes.

---

### Common Mistakes

- Assuming Docker changes application logic.
- Forgetting persistence.
- Ignoring metadata filtering.

---

## Topic 4: How Does Qdrant Search Millions of Vectors?

### Why Do We Need It?

Searching every vector is acceptable for hundreds of vectors but becomes extremely slow for millions.

---

### What Is It?

Qdrant first builds an **Index**.

Instead of checking every vector, it searches only the most promising vectors.

Qdrant implements this using **HNSW (Hierarchical Navigable Small World)**.

---

### Example

Imagine a book.

Instead of reading every page, you first open the relevant chapter.

The chapter contains only related pages.

Similarly, Qdrant searches only the relevant region of the index.

---

### How Does It Work?

Step 1

Embeddings are inserted.

Step 2

Qdrant builds an HNSW graph.

Step 3

The query enters the graph.

Step 4

The graph navigates through nearby vectors.

Step 5

The nearest neighbors are returned.

---

### Real-World Issues, Edge Cases & Debugging

- Missing index causes brute-force search.
- Incorrect HNSW parameters reduce recall.
- Large collections require parameter tuning.

---

### Common Mistakes

- Assuming HNSW compares every vector.
- Confusing HNSW with metadata indexes.

## Topic 5: Where Does Qdrant Store Vectors?

### Why Do We Need It?

After understanding how Qdrant searches vectors, the next question is:

**Where are these vectors actually stored?**

During development we need a lightweight setup, while production requires persistent storage.

---

### What Is It?

Qdrant supports two storage modes:

- In-Memory Storage
- Persistent Storage

In-Memory stores vectors in RAM.

Persistent Storage stores vectors on disk.

---

### Example

Development

```python
QdrantClient(":memory:")
```

Production

```python
QdrantClient(url="http://localhost:6333")
```

---

### How Does It Work?

Step 1

Documents are converted into embeddings.

Step 2

Embeddings are stored in RAM when using `:memory:`.

Step 3

Restarting Python removes all vectors.

Step 4

Production stores vectors on disk using Docker.

Step 5

Vectors remain available even after restarting the application.

---

### Real-World Issues, Edge Cases & Debugging

- Restarting Python removes all vectors in In-Memory mode.
- Missing Docker volumes cause permanent data loss.
- Disk storage is required for production systems.

---

### Common Mistakes

- Using `:memory:` in production.
- Forgetting Docker volumes.
- Assuming RAM and disk behave the same.

---

## Topic 6: How Does Qdrant Search Only Relevant Documents?

### Why Do We Need It?

Similarity search alone searches the entire collection.

In production, we often need to search only a subset of documents.

Example:

- Only FAQs
- Only FD documents
- Only English documents

---

### What Is It?

Metadata is additional information stored with every document.

Qdrant stores metadata as **Payload** and uses it to filter documents before similarity search.

---

### Example

Document

```text
Text:
FD Interest Rate

Payload:
doc_type = faq
product = FD
language = English
```

Filter

```text
doc_type = faq
product = FD
```

Only FD FAQ documents are searched.

---

### How Does It Work?

Step 1

Store metadata during document ingestion.

Step 2

Create payload indexes for frequently filtered fields.

Step 3

User sends a query.

Step 4

Qdrant filters documents using payload.

Step 5

Similarity search runs only on the filtered documents.

Step 6

Relevant documents are returned.

---

### Real-World Issues, Edge Cases & Debugging

- Wrong payload field returns zero results.
- Missing metadata excludes documents.
- No payload index slows filtering.
- Too many filters may return very few documents.

---

### Common Mistakes

- Filtering in Python after retrieval.
- Forgetting to store metadata.
- Using inconsistent metadata values.
- Not creating payload indexes.

---

## Topic 7: How Do We Secure Customer Data?

### Why Do We Need It?

Semantic search can accidentally return another customer's personal information.

Customer data must always be protected.

---

### What Is It?

Qdrant secures customer data using:

- Separate collections.
- Caller-scoped filtering.

Only the authenticated customer's records are searched.

---

### Example

Authenticated Customer

```text
fd_no = BJ2019FD7717
```

Only this customer's records are searched.

Policy documents remain in a separate collection.

---

### How Does It Work?

Step 1

Store policy documents in one collection.

Step 2

Store customer records in another collection.

Step 3

Authenticate the customer.

Step 4

Apply caller-scoped filtering.

Step 5

Search only the authenticated customer's records.

Step 6

Return only authorized data.

---

### Real-World Issues, Edge Cases & Debugging

- Cross-customer data leakage.
- Mixing policy and customer documents.
- Deleting customer data only from the source database.
- Missing caller filter exposes other customers.

---

### Common Mistakes

- Storing all documents in one collection.
- Forgetting caller-scoped filtering.
- Assuming metadata filtering alone provides complete security.
- Forgetting to delete vectors from Qdrant.

---

## Topic 8: What Happens When the Embedding Model Changes?

### Why Do We Need It?

Over time, better embedding models become available.

Changing the embedding model changes every embedding in the collection.

---

### What Is It?

Embedding Model Migration is the process of rebuilding the Vector Database using embeddings generated by a new embedding model.

Old and new embeddings cannot be mixed.

---

### Example

Old Model

```text
paraphrase-multilingual-MiniLM-L12-v2
```

New Model

```text
all-MiniLM-L6-v2
```

The document text remains the same.

Only the embeddings are regenerated.

---

### How Does It Work?

Step 1

Load the new embedding model.

Step 2

Generate new embeddings for every document.

Step 3

Create a new Qdrant collection.

Step 4

Store the new embeddings.

Step 5

Validate search quality.

Step 6

Update the alias to use the new collection.

Step 7

Delete the old collection after verification.

---

### Real-World Issues, Edge Cases & Debugging

- Changing the model without rebuilding the collection.
- Mixing embeddings from different models.
- Different embedding dimensions.
- Manifest detects file changes but not model changes.

---

### Common Mistakes

- Changing the model constant without re-indexing.
- Updating the existing collection instead of creating a new one.
- Forgetting alias validation.
- Not tracking the embedding model in the manifest.